# T2 — MLflow: experimentos, parâmetros e métricas

**Objetivo:** parar de anotar resultados em planilha. Cada treino vira um *run* registrado,
comparável e reproduzível.

| Termo | O que é |
|---|---|
| **Experimento** | agrupa runs de um mesmo problema (`aula01-experimentos`) |
| **Run** | uma execução de treino |
| **Parâmetro** | entrada do run (hiperparâmetros, tipo de modelo) |
| **Métrica** | resultado numérico (RMSE, MAE, R²) |
| **Tag** | metadado livre (autor, versão dos dados, commit) |
| **Artefato** | arquivo gerado (gráfico, modelo serializado) |

> Na T1, o `train.py` do pipeline já registrou runs no experimento `bike-demand-pipeline`.
> Agora criamos os nossos à mão, em outro experimento, para entender cada peça.

In [1]:
import os, sys
from pathlib import Path

# Trabalha a partir da raiz do repositório (a pasta que contém o params.yaml)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "params.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import logging
import mlflow
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from bike import load_raw, regression_metrics, time_split, tracking_uri, xy

pd.set_option("display.width", 160)
mlflow.set_tracking_uri(tracking_uri())          # onde está o servidor
mlflow.set_experiment("aula01-experimentos")     # cria (se não existir) e ativa
print("MLflow em", mlflow.get_tracking_uri())

2026/11/04 21:25:46 INFO mlflow.tracking.fluent: Experiment with name 'aula01-experimentos' does not exist. Creating a new experiment.


MLflow em http://127.0.0.1:5000


---
# Parte 1 · Juntos

## 1. Dados e separação temporal

Os últimos 28 dias viram teste. Em série temporal **não se embaralha**: o modelo treina no passado
e é avaliado no futuro, como acontece em produção.

In [2]:
df = load_raw()
train, test = time_split(df, test_days=28)
X_train, y_train = xy(train)
X_test, y_test = xy(test)

print(f"treino: {len(train):,} linhas até {train.dteday.max():%d/%m/%Y}")
print(f"teste : {len(test):,} linhas de {test.dteday.min():%d/%m/%Y} a {test.dteday.max():%d/%m/%Y}")

treino: 7,976 linhas até 03/12/2011
teste : 669 linhas de 04/12/2011 a 31/12/2011


## 2. Primeiro run, feito à mão: o baseline

Todo projeto precisa de uma régua. Aqui, uma regressão linear.

In [3]:
from sklearn.linear_model import LinearRegression

with mlflow.start_run(run_name="baseline-linear") as run:
    model = LinearRegression()
    model.fit(X_train, y_train)
    metrics = regression_metrics(y_test, model.predict(X_test))

    mlflow.log_param("model_type", "LinearRegression")   # parâmetro: entrada do run
    mlflow.log_metrics(metrics)                           # métricas: resultado
    mlflow.set_tag("autor", "turma")                      # tag: metadado livre
    mlflow.sklearn.log_model(model, name="model",         # artefato: o modelo
                             serialization_format="cloudpickle")

print(run.info.run_id, metrics)

2026/11/04 21:25:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run baseline-linear at: http://127.0.0.1:5000/#/experiments/2/runs/be607d348e69437882b064f3db69b1e4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
be607d348e69437882b064f3db69b1e4 {'rmse': 94.00358717979029, 'mae': 72.29530552566453, 'r2': 0.18304008429127294}


➡️ **Abra a interface do MLflow** (porta 5000) → experimento `aula01-experimentos` → clique no run `baseline-linear`.
A página do run mostra as **métricas**, os **parâmetros** e as **tags**. O modelo fica em **Logged models**, no fim da página:
clique em `model` e abra a aba **Artifacts** para ver o arquivo `MLmodel`. Ele descreve como carregar o modelo depois
(o "manual de instruções" do artefato).

> 💬 No MLflow 3, o modelo é uma entidade própria, ligada ao run. Por isso a aba *Artifacts* do run fica vazia aqui;
> nos próximos runs, ela guarda o gráfico.

> 💬 **Sobre o aviso amarelo (WARNING) na saída da célula:** o MLflow avisa que *pickle* pode executar código ao ser carregado.
> O formato padrão novo (*skops*) é mais seguro, mas exige listar "tipos confiáveis" para modelos de árvore.
> Como os modelos são nossos, seguimos com pickle. Segurança de artefatos é tema da Aula 4.

## 3. Uma função de treino reutilizável (com gráfico como artefato)

Escrever o `with mlflow.start_run()` toda vez é repetitivo. A função abaixo faz o mesmo que a
célula anterior e ainda anexa um gráfico real × previsto. **É ela que você vai usar na Parte 2.**

In [4]:
logging.getLogger("mlflow.sklearn").setLevel(logging.ERROR)  # já entendemos o aviso do pickle

ts_test = test["dteday"] + pd.to_timedelta(test["hr"], unit="h")


def plot_pred_vs_real(y_pred, title):
    fig, ax = plt.subplots(figsize=(11, 3))
    ax.plot(ts_test, y_test.values, label="real", lw=1)
    ax.plot(ts_test, y_pred, label="previsto", lw=1)
    ax.set_title(title)
    ax.legend()
    return fig


def train_and_log(model_cls, params, run_name=None, tags=None):
    """Treina um modelo e registra tudo no MLflow. Devolve (run_id, métricas)."""
    with mlflow.start_run(run_name=run_name or model_cls.__name__) as run:
        model = model_cls(**params).fit(X_train, y_train)
        y_pred = model.predict(X_test)
        metrics = regression_metrics(y_test, y_pred)

        mlflow.set_tags(tags or {})
        mlflow.log_param("model_type", model_cls.__name__)
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        fig = plot_pred_vs_real(y_pred, f"{model_cls.__name__} | RMSE = {metrics['rmse']:.1f}")
        mlflow.log_figure(fig, "pred_vs_real.png")
        plt.close(fig)
        mlflow.sklearn.log_model(model, name="model", serialization_format="cloudpickle",
                                 input_example=X_train.head(3))
    return run.info.run_id, metrics

## 4. Usando a função: duas florestas aleatórias

In [5]:
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor

for max_depth in [8, 14]:
    params = {"n_estimators": 100, "max_depth": max_depth, "n_jobs": -1, "random_state": 42}
    run_id, m = train_and_log(RandomForestRegressor, params,
                              run_name=f"rf-depth{max_depth}", tags={"autor": "turma"})
    print(f"RandomForest max_depth={max_depth:2d}  RMSE = {m['rmse']:5.1f} | R² = {m['r2']:.3f}")

🏃 View run rf-depth8 at: http://127.0.0.1:5000/#/experiments/2/runs/d99c7a981c3043319ec355b5de69e3c1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
RandomForest max_depth= 8  RMSE =  60.0 | R² = 0.667


🏃 View run rf-depth14 at: http://127.0.0.1:5000/#/experiments/2/runs/9983f2460f604376b8defbcf269b5a06
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
RandomForest max_depth=14  RMSE =  56.8 | R² = 0.702


## 5. Comparar runs

Pela interface: marque os runs → **Compare** (gráficos de coordenadas paralelas e dispersão).
Por código: `mlflow.search_runs` devolve um DataFrame.

In [6]:
runs = mlflow.search_runs(
    experiment_names=["aula01-experimentos"],
    filter_string="tags.autor = 'turma' and metrics.rmse > 0",
    order_by=["metrics.rmse ASC"],
)
cols = ["tags.mlflow.runName", "params.model_type", "params.max_depth", "metrics.rmse", "metrics.r2"]
display(runs[cols])

,tags.mlflow.runName,params.model_type,params.max_depth,metrics.rmse,metrics.r2
0,rf-depth14,RandomForestRegressor,14,56.816898,0.701553
1,rf-depth8,RandomForestRegressor,8,59.981820,0.667378
2,baseline-linear,LinearRegression,NaN,94.003587,0.183040


## 6. Autolog: o MLflow registra sozinho

Uma linha liga o registro automático de parâmetros, métricas de treino e modelo.
Métricas calculadas no teste (como `model.score`) também são capturadas.

In [7]:
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="hgb-autolog"):
    model = HistGradientBoostingRegressor(learning_rate=0.05, max_iter=500, random_state=42)
    model.fit(X_train, y_train)
    model.score(X_test, y_test)   # R² no teste, registrado automaticamente

mlflow.sklearn.autolog(disable=True)

🏃 View run hgb-autolog at: http://127.0.0.1:5000/#/experiments/2/runs/5de64a3374de4ba48eee900dd924535c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


➡️ Na interface, compare a quantidade de parâmetros registrados pelo autolog com a dos runs manuais.

---
# Parte 2 · Sozinhos

Agora é com você: o mesmo fluxo, com outro modelo. Complete os `# TODO` e rode.
Os seus runs levam a tag `autor = "aluno"` (os da Parte 1 são `"turma"`).

### 1. Treine 3 variações de `HistGradientBoostingRegressor`

Use a `train_and_log`, como fizemos com a floresta aleatória. Mude `learning_rate`, `max_iter` e `max_leaf_nodes`.

In [8]:
grid = [
    {"learning_rate": 0.05, "max_iter": 500, "max_leaf_nodes": 31, "random_state": 42},
    {"learning_rate": 0.10, "max_iter": 300, "max_leaf_nodes": 63, "random_state": 42},
    {"learning_rate": 0.20, "max_iter": 200, "max_leaf_nodes": 15, "random_state": 42},
]

for params in grid:
    run_id, m = train_and_log(HistGradientBoostingRegressor, params,
                              run_name="hgb", tags={"autor": "aluno"})
    print(params, "-> RMSE", round(m["rmse"], 1))

🏃 View run hgb at: http://127.0.0.1:5000/#/experiments/2/runs/78027d9da4474d65af76a75b331f642c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
{'learning_rate': 0.05, 'max_iter': 500, 'max_leaf_nodes': 31, 'random_state': 42} -> RMSE 54.7


🏃 View run hgb at: http://127.0.0.1:5000/#/experiments/2/runs/c6681a3135494199ade2faa8597672c9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
{'learning_rate': 0.1, 'max_iter': 300, 'max_leaf_nodes': 63, 'random_state': 42} -> RMSE 57.3


🏃 View run hgb at: http://127.0.0.1:5000/#/experiments/2/runs/2abefddfa9b04038986927881d26e905
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
{'learning_rate': 0.2, 'max_iter': 200, 'max_leaf_nodes': 15, 'random_state': 42} -> RMSE 54.2


### 2. Encontre o seu melhor run

Primeiro **pela interface** (filtre por `tags.autor = 'aluno'` e ordene pelo RMSE). Depois **por código**,
como no passo 5 da Parte 1.

In [9]:
meus = mlflow.search_runs(
    experiment_names=["aula01-experimentos"],
    filter_string="tags.autor = 'aluno' and metrics.rmse > 0",
    order_by=["metrics.rmse ASC"],
)
display(meus[["tags.mlflow.runName", "params.learning_rate", "params.max_iter",
              "params.max_leaf_nodes", "metrics.rmse"]])
meu_melhor = meus.iloc[0]

,tags.mlflow.runName,params.learning_rate,params.max_iter,params.max_leaf_nodes,metrics.rmse
0,hgb,0.2,200,15,54.159514
1,hgb,0.05,500,31,54.672478
2,hgb,0.1,300,63,57.273812


### 3. Quanto você melhorou em relação ao baseline?

In [10]:
baseline = mlflow.search_runs(experiment_names=["aula01-experimentos"],
                              filter_string="tags.mlflow.runName = 'baseline-linear'").iloc[0]
ganho = 1 - meu_melhor["metrics.rmse"] / baseline["metrics.rmse"]
print(f"Seu melhor: RMSE {meu_melhor['metrics.rmse']:.1f} | baseline: {baseline['metrics.rmse']:.1f} "
      f"| ganho: {ganho:.0%}")

Seu melhor: RMSE 54.2 | baseline: 94.0 | ganho: 42%


---
## Para discutir
1. No seu melhor run, aponte um exemplo de **parâmetro**, de **métrica**, de **tag** e de **artefato**.
2. O que o **autolog** registrou a mais que os runs manuais? Você usaria autolog em produção?
3. Por que **não embaralhamos** treino e teste neste problema?

**Respostas esperadas:**
1. Parâmetro: `learning_rate = 0.2` (entrada). Métrica: `rmse` (resultado). Tag: `autor` (metadado livre).
   Artefato: `pred_vs_real.png` e a pasta do modelo.
2. Todos os hiperparâmetros do estimador, métricas de treino e o modelo. Não esquece nada, mas gera ruído e
   usa nomes próprios para as métricas. Bom para explorar; em produção, registre explicitamente o que importa.
3. É uma série temporal. Embaralhar deixaria o modelo "ver o futuro" (vazamento) e a métrica ficaria otimista.
   Em produção ele sempre prevê o futuro.